In [1]:
from chat_client import *

In [2]:
# Create the agents
local_agent = ChatAgent(
    chat_client=chat_client,
    instructions=(
        "You are a helpful assistant that can suggest authentic and interesting local activities "
        "or places to visit for a user and can utilize any context information provided."
    ),
    name="local_agent",
    description="A local assistant that can suggest local activities or places to visit.",
)

In [3]:
language_agent = ChatAgent(
    chat_client=chat_client,
    instructions=(
        "You are a helpful assistant that can review travel plans, providing feedback on important/critical "
        "tips about how best to address language or communication challenges for the given destination. "
        "If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale."
    ),
    name="language_agent",
    description="A helpful assistant that can provide language tips for a given destination.",
)


In [4]:
travel_summary_agent = ChatAgent(
    chat_client=chat_client,
    instructions=(
        "You are a helpful assistant that can take in all of the suggestions and advice from the other agents "
        "and provide a detailed final travel plan. You must ensure that the final plan is integrated and complete. "
        "YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. Provide a comprehensive summary when all perspectives "
        "from other agents have been integrated."
    ),
    name="travel_summary_agent",
    description="A helpful assistant that can summarize the travel plan.",
)

In [5]:
# Event callback for streaming output with rich formatting
async def on_event(event: MagenticCallbackEvent) -> None:
    if isinstance(event, MagenticOrchestratorMessageEvent):
        emoji = "✅" if event.kind == "task_ledger" else "🦠"
        print(
                Markdown(event.message.text),
                title=f"{emoji} orchestrator: {event.kind}",
                border_style="bold green",
                padding=(1, 2),
            )
        
    elif isinstance(event, MagenticAgentMessageEvent):
        print(
            
                Markdown(event.message.text),
                title=f"🤖 {event.agent_id}",
                border_style="bold blue",
                padding=(1, 2),
            )
       

In [6]:
magentic_orchestrator = (
    MagenticBuilder()
    .participants(
        local_agent=local_agent,
        language_agent=language_agent,
        travel_summary_agent=travel_summary_agent,
    )
    .on_event(on_event, mode=MagenticCallbackMode.NON_STREAMING)
    .with_standard_manager(
        chat_client=chat_client,
        max_round_count=20,
        max_stall_count=3,
        max_reset_count=2,
    )
    .build()
)

Cycle detected in the workflow graph involving: agent_travel_summary_agent -> agent_language_agent -> agent_local_agent -> magentic_orchestrator -> agent_travel_summary_agent. Ensure termination or iteration limits exist.


In [7]:
async for event in magentic_orchestrator.run_stream("Plan a half-day trip to Costa Rica"):
        if isinstance(event, WorkflowOutputEvent):
            final_result = event.data
            print(
                "Final Travel Plan:\n",
                final_result.text,
            )

Final Travel Plan:
 For a half-day trip in Costa Rica, you have some fantastic options to choose from depending on your starting point:

1. **From San José**:
   - **La Paz Waterfall Gardens**: Enjoy beautiful waterfalls, hiking trails, and a variety of wildlife. It’s about an hour and a half from San José.
   - **Poás Volcano National Park**: Visit the stunning crater of this active volcano, located less than two hours away.
   - **Doka Coffee Estate**: Dive into the process of Costa Rican coffee production on this engaging tour, just a short drive away.

2. **From Coastal Areas like Jaco or Quepos**:
   - **Carara National Park**: Explore transitional ecosystems and spot diverse wildlife, including crocodiles and exotic birds. Just over an hour’s drive from Jaco.
   - **Manuel Antonio Waterfall**: Near Quepos, take a refreshing swim in a jungle setting at these local waterfalls.

These destinations offer a rich taste of Costa Rica’s natural beauty and cultural heritage, perfect for a